In [ ]:
from pathlib import Path
from typing import Literal
import warnings

import arviz_plots as azp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

from compressor_fouling_modeling.utility import (
    calculate_data_masks,
    plot_timeseries_grid,
    prepare_model_input,
    visualize_correlations,
    visualize_imputation,
)

azp.style.use("dark_background")  # pick style of interest
%config InlineBackend.figure_format = 'retina'  # high resolution figures
warnings.filterwarnings("ignore")

In [ ]:
RANDOM_SEED = 14
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
# Define project root relative to notebook location
PROJECT_ROOT = Path().resolve().parents[0]  # goes up one level from /notebooks/
IMAGE_DIR = Path(PROJECT_ROOT / "results" / "plots")
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
# Check if the directory exists and create it if it doesn't
if not IMAGE_DIR.exists():
    try:
        IMAGE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Directory '{IMAGE_DIR}' created successfully.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"Directory '{IMAGE_DIR}' already exists.")

In [ ]:
# Load file
data_raw = pd.read_csv(DATA_DIR / "raw" / "ds_compressor_data.csv")
data_raw["timestamp"] = pd.to_datetime(data_raw["timestamp"], utc=True)
data_raw = data_raw.set_index("timestamp").sort_index()
print(f"Data covers period of {data_raw.index.min()} to {data_raw.index.max()} with {data_raw.index.diff().mean()} frequency.")

In [ ]:
data_raw.iloc[:, 3].astype(float)

In [ ]:
# convert (and overwrite) int datatype to float to avoid implicit rounding with NumPy arrays or even errors with newer pandas versions
data_raw.isetitem(3, data_raw.iloc[:, 3].astype(float))
data_raw.isetitem(4, data_raw.iloc[:, 4].astype(float))

Off_Gas_Fraction_Percentage: This represents the percentage of off-gas (or residual gas) in the system. Off-gas is the gas that is not fully compressed or utilized and can affect the efficiency and output of the compressor.

Outlet_Flow_Rate_SP: This stands for the Set Point (SP) for the outlet flow rate. A set point is a target value that the system aims to maintain. In this case, it's the desired flow rate of the gas or fluid exiting the compressor.

Outlet_Pressure_SP: Similar to the outlet flow rate SP, this is the set point for the pressure at the compressor's outlet. It's the target pressure that the system strives to achieve.

Outlet_Flow_Rate: This is the actual flow rate of the gas or fluid exiting the compressor. It can be compared to the set point (Outlet_Flow_Rate_SP) to evaluate the compressor's performance.

Outlet_Pressure: This column records the actual pressure of the gas or fluid at the compressor's outlet. It can be compared to the set point (Outlet_Pressure_SP) to assess how well the compressor is maintaining the desired pressure.


In [ ]:
type(Path(IMAGE_DIR / "data_time_series.png"))

In [ ]:
plot_config = [
    [("Off_Gas_Fraction_Percentage", "b")],
    [("Ambient_Humidity", "y")],
    [("Outlet_Temperature", "g"), ("Inlet_Temperature", "c")],
    [("Inlet_Flow_Rate", "y"), ("Outlet_Flow_Rate", "b"), ("Outlet_Flow_Rate_SP", "r")],
    [("Inlet_Pressure", "y"), ("Outlet_Pressure", "b"), ("Outlet_Pressure_SP", "r")],
]
fname = Path(IMAGE_DIR / "data_time_series.png")
plot_timeseries_grid(data_raw, plot_config, save=True, fname=fname)

In [ ]:
data_nonzero_setpoints = data_raw[data_raw["Outlet_Flow_Rate"] > 0]
fig, ax = plt.subplots(1, 1, figsize=(15, 7))
sns.heatmap(data_nonzero_setpoints.isna(), cbar=False, ax=ax)
display(data_nonzero_setpoints.isna().mean().sort_values(ascending=False))
plt.show()
plt.close(fig)
del fig, ax

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 7))
visualize_imputation(axs[0], data_raw, feature="Outlet_Temperature")
visualize_imputation(axs[1], data_raw, feature="Inlet_Temperature")
visualize_imputation(axs[2], data_raw, feature="Off_Gas_Fraction_Percentage")
visualize_imputation(axs[3], data_raw, feature="Outlet_Pressure")
plt.show()
plt.close(fig)
del fig, axs

Avoid: Mean/median imputation - destroys temporal relationships in process data. The time method does a better job by simply interpolating missing values within number of consecutive NaNs to fill. I initially drop rows with missing Outlet_Pressure as the target with limited missing values and replace missing values through simple interpolate method ('time') that preserves distributions (temporal relationships) to a great extent.  

In [ ]:
data, X, y = prepare_model_input(data_raw, None, feature_engineering_allowed=True)

In [ ]:
baseline_mask, shutin_mask = calculate_data_masks(
    data
)
X_baseline = X.loc[baseline_mask]
y_baseline = y.loc[baseline_mask]

In [ ]:
baseline = [("2024-08-01", "2024-11-03"), ("2025-01-01", "2025-02-09")]
fig, ax = plt.subplots(1, 1, figsize=(15, 7))
ax.plot(baseline_mask * 1)
for start, end in baseline:
    ax.axvspan(start, end, color="red", alpha=0.3)
plt.show()
plt.close(fig)
del fig, ax

### Investigate correlations

In [ ]:
data_baseline = data.loc[baseline_mask]

In [ ]:
correlation_methods: list[Literal["pearson", "kendall", "spearman"]] = [
    "pearson",
    "kendall",
    "spearman",
]
for corr_method in correlation_methods:
    fig, ax = plt.subplots(1, 1, figsize=(15, 7))
    sns.heatmap(data_baseline.corr(method=corr_method), annot=True, fmt=".2f", ax=ax)
    print(corr_method)
    display(
        data_baseline.corr(method=corr_method)["Outlet_Pressure"].sort_values(
            ascending=False, key=abs
        )
    )
    plt.show()
    plt.close(fig)
    del fig, ax

In [ ]:
scaler = StandardScaler()
data_baseline_scaled = pd.DataFrame(
    scaler.fit_transform(data_baseline),
    columns=data_baseline.columns,
    index=data_baseline.index,
)
target_mask = data_baseline_scaled.columns != "Outlet_Pressure"
mi_scores = mutual_info_regression(
    data_baseline_scaled.loc[:, target_mask],
    data_baseline_scaled["Outlet_Pressure"],
)
mi_series = pd.Series(
    mi_scores, index=data_baseline_scaled.columns[target_mask]
).sort_values(ascending=False)
print("Mutual Information Scores:")
print(mi_series)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(20, 15))
visualize_correlations(
    axs[0, 0], data_baseline, "Inlet_Temperature", "Outlet_Temperature"
)
visualize_correlations(axs[0, 1], data_baseline, "Inlet_Flow_Rate", "Outlet_Flow_Rate")
visualize_correlations(axs[0, 2], data_baseline, "Inlet_Flow_Rate", "Inlet_Pressure")
visualize_correlations(
    axs[1, 0], data_baseline, "Outlet_Temperature", "Outlet_Pressure"
)
visualize_correlations(axs[1, 1], data_baseline, "Outlet_Flow_Rate", "Outlet_Pressure")
# non-linear variable
visualize_correlations(
    axs[1, 2], data_baseline, "Inlet_Pressure_x_Flow", "Outlet_Pressure", aspect="auto"
)

plt.show()
plt.close(fig)
del fig, axs

# save data for analysis

In [ ]:
X_baseline.to_csv(DATA_DIR / "processed" / "X_baseline.csv")
y_baseline.to_csv(DATA_DIR / "processed" / "y_baseline.csv")
X.to_csv(DATA_DIR / "processed" / "X_full.csv")
y.to_csv(DATA_DIR / "processed" / "y_full.csv")
baseline_mask.to_csv(DATA_DIR / "processed" / "baseline_mask.csv")
shutin_mask.to_csv(DATA_DIR / "processed" / "shutin_mask.csv")

### Strategy: "Expected vs. Actual" (Residual Analysis)

The core challenge, as stated, is that "outlet pressure is influenced by multiple operating parameters." Therefore, we cannot simply set a hard threshold (e.g., "Alert if pressure < 50 psi") because the pressure changes based on the set point (SP) and inlet conditions.

Since we need to isolate the fouling effect from normal operating changes, the best approach is to build a Virtual Sensor (Regression Model) which solves the specific "Isolating Effects" problem by including Setpoints, Flow, Inlet Pressure, Temperature, ... as inputs. The model "knows" that pressure should drop if flow drops. It won't flag that as a fouling status. It will only flag when pressure drops unaccountably. We train a model to predict what the Outlet Pressure should be given the current inlet conditions and set points. In other words, we need a baseline estimator. Since the machine is controlled, Outlet Pressure ~ Setpoint is over 90% of the answer. We need a linear regression to adjust that baseline slightly based on other operating variables:
- $\text{Output} \approx \text{Setpoint} + \text{Efficiency\_Correction}$

Hypothesis: If the machine is clean, Actual Pressure ≈ Predicted Pressure.
Fouling Signal: If the machine is fouling, Actual Pressure < Predicted Pressure. The gap between them (the residual) is the fouling indicator.


The Solution: Take very important note that we treat this as a Physics Problem, not a Forecasting Problem. We are building a Virtual Sensor, not a Stock Market Predictor.

- Forecasting: Predicting $Y_{t+1}$ based on $Y_{t}$ (Time matters strictly).
- Virtual Sensing: Predicting $Y_{t+1}$ based on $X_t$ (Physics matters).

We use the classical regression model, to only allow the dependent variable to be influenced by current values of the
independent variables (The model is "blind" to the past).
The fact that we broke the autocorrelation link (by removing lags) is exactly what makes this an anomaly detector.

Imagine the Fouling Scenario (January 2025):

1. Inputs (X): Setpoint is still 100. Flow is still 50.
2. The "Physics" Model: Looks at X. Says: "Based on Flow 50 and SP 100, the pressure MUST be 100."
3. The Reality (Y): The machine is fouled. Actual pressure is 80.
4. The Residual: 100 − 80 = +20 (Anomaly Detected!)

Contrast this with the "Autocorrelated/Forecasting" Model:

1. Inputs $Y_{t−1}$: The pressure 1 day ago was 100.
2. The "Lazy" Model: Looks at $Y_{t−1}$ and says: "Since it was 100 a day ago, it's probably 100 now."
3. The Residual: 100 − 100 = 0 (Anomaly Missed!)

Note that we have 4 pressure and flow setpoints (disregard the shutin period). The physics of the compressor $ \text{Pressure} = f\left(\text{Setpoint}, \text{Flow}, \text{Temp}, \dots \right) $ does not change from day to day (data sample intervals) and from setpoint to setpoint. Therefore, we shuffle the data for validation (and to break autocorrelation) as we want to assure model has learned pure physics. This ensures that every training fold contains a mix of all the four setpoints
.

### Summary for the Stakeholder

"We will build a digital twin of the compressor using historical 'healthy' data. This model will predict what the outlet pressure should be based on current operating conditions (flow, temperature, inlet pressure). We will then track the deviation between the model's prediction and the actual sensor reading. A growing deviation indicates fouling, allowing us to alert maintenance before efficiency drops below critical levels."